# SCFP experiments on a GPU

Two experiments; choose one in the next cell:

* `signals` (recommended first): builds a natural, **unfiltered** problem set and tests generator-side signals (self-consistency, log-probs, critique verdicts, hidden-state probes) against text-only TF-IDF, on the natural set and on SCFP v1. About 6-10 h on a T4.
* `larger_models`: DeBERTa-v3-large, RoBERTa-large and DeBERTa-v3-small with 5-fold cross-validation on SCFP v1. About 5-7 h on a T4.

**Kaggle:** upload `SCFP_GPU_package.zip` as a Dataset, add it to this notebook, set *Accelerator* to *GPU T4 x1* (or P100) and *Internet* on, then *Save & Run All*.
**Colab:** *Runtime > Change runtime type > T4 GPU*, run all cells, upload `SCFP_GPU_package.zip` when asked.

Both runs are resumable: if a session ends, run again with the previous results zip next to the package.

In [ ]:
EXPERIMENT = 'signals'   # 'signals' or 'larger_models'


In [ ]:
import os, glob, shutil, subprocess, zipfile
IN_KAGGLE = os.path.exists('/kaggle')
WORK = '/kaggle/working' if IN_KAGGLE else '/content'
os.chdir(WORK)
if IN_KAGGLE:
    found = glob.glob('/kaggle/input/**/run_gpu.sh', recursive=True) or glob.glob('/kaggle/input/**/SCFP_GPU_package.zip', recursive=True)
    assert found, 'Add the SCFP_GPU_package dataset to this notebook'
    src = found[0]
    if src.endswith('.zip'):
        zipfile.ZipFile(src).extractall(WORK)
    else:
        shutil.copytree(os.path.dirname(src), os.path.join(WORK, 'SCFP_GPU_package'), dirs_exist_ok=True)
else:
    from google.colab import files
    if not os.path.exists('SCFP_GPU_package'):
        up = files.upload()  # choose SCFP_GPU_package.zip
        zipfile.ZipFile(next(iter(up))).extractall(WORK)
# resume: if a previous results zip is present, restore finished runs
prev = glob.glob(os.path.join(WORK, 'scfp_*results*.zip')) + glob.glob('/kaggle/input/**/scfp_*results*.zip', recursive=True)
if prev:
    zipfile.ZipFile(prev[0]).extractall(os.path.join(WORK, 'SCFP_GPU_package'))
    print('restored previous results from', prev[0])
print(os.listdir(os.path.join(WORK, 'SCFP_GPU_package')))

In [ ]:
!nvidia-smi
!pip -q install -r {WORK}/SCFP_GPU_package/requirements_gpu.txt

In [ ]:
SCRIPT = 'run_signals.sh' if EXPERIMENT == 'signals' else 'run_gpu.sh'
!cd {WORK}/SCFP_GPU_package && bash {SCRIPT} 2>&1 | grep -vE 'it/s\]|Loading weights'


In [ ]:
R = f'{WORK}/SCFP_GPU_package'
if EXPERIMENT == 'signals':
    print(open(f'{R}/benchmark/scfp_runs/signals_v2/results/improve_summary.md').read())
    out = f'{R}/scfp_signals_results.zip'
else:
    print(open(f'{R}/benchmark/scfp_runs/scfp_v1/results/cv_summary.md').read())
    p = f'{R}/benchmark/scfp_runs/scfp_v1/results/fixed_split_summary.md'
    if os.path.exists(p): print(open(p).read())
    out = f'{R}/scfp_gpu_results.zip'
if IN_KAGGLE:
    shutil.copy(out, WORK); print('Download', os.path.basename(out), 'from the Output panel.')
else:
    from google.colab import files; files.download(out)
